<a href="https://colab.research.google.com/github/Reileen00/Monte-Carlo-Random-Walk/blob/main/Monte_Carlo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install gymnasium
!pip install numpy
!pip install math

ERROR: Could not find a version that satisfies the requirement math (from versions: none)
ERROR: No matching distribution found for math


In [15]:
!pip install swig
!pip install "gymnasium[box2d]"

In [13]:
import gymnasium as gym
import numpy as np
import math
from torch.utils.tensorboard import SummaryWriter
from gymnasium.wrappers import RecordVideo

In [14]:
#Create training environment
env = gym.make("LunarLander-v3")
writer = SummaryWriter(log_dir = "runs/lunarlander_mc")

In [16]:
#Discretization bins for 8-dimensional state
NUM_BINS = (8,8,8,8,8,8,2,2)
TOTAL_STATES = np.prod(NUM_BINS)

In [17]:
#Pre-defined observation bounds
obs_space_low = np.array([-1.5,-1.5,-1.5,-2.0,-math.pi,-5.0,0.0,0.0])
obs_space_high = np.array([1.5,1.5,1.5,2.0,math.pi,5.0,1.0,1.0])
obs_range = obs_space_high - obs_space_low

In [18]:
#Discretize continous state
def discretize(obs):
  ratios = np.clip((obs-obs_space_low)/obs_range,0,0.999)
  bins = (ratios * NUM_BINS).astype(int)

  #Convert to single index
  multipliers = np.array([NUM_BINS[1] * NUM_BINS[2] * NUM_BINS[3] * NUM_BINS[4] * NUM_BINS[5] * NUM_BINS[6] * NUM_BINS[7],
                          NUM_BINS[2] * NUM_BINS[3] * NUM_BINS[4] * NUM_BINS[5] * NUM_BINS[6] * NUM_BINS[7],
                          NUM_BINS[3] * NUM_BINS[4] * NUM_BINS[5] * NUM_BINS[6] * NUM_BINS[7],
                          NUM_BINS[4] * NUM_BINS[5] * NUM_BINS[6] * NUM_BINS[7],
                          NUM_BINS[5] * NUM_BINS[6] * NUM_BINS[7],
                          NUM_BINS[6] * NUM_BINS[7],
                          NUM_BINS[7],
                          1
                          ])
  return np.dot(bins,multipliers)


In [19]:
#Initialize Q-table and visit counts for Monte Carlo
Q = np.zeros((TOTAL_STATES , env.action_space.n))
returns_sum = np.zeros((TOTAL_STATES,env.action_space.n))
returns_count = np.zeros((TOTAL_STATES, env.action_space.n))

In [20]:
#Hyperparameters
GAMMA = 0.99
EPSILON = 1.0
EPSILON_MIN = 0.05
EPSILON_DECAY = 0.9995
NUM_EPISODES = 30000

In [21]:
#Reward tracking
all_rewards = []

In [22]:
#Epsilon-greedy policy
def choose_action(state,epsilon):
  if np.random.rand() < epsilon:
    return np.random.randint(env.action_space.n)
  return np.argmax(Q[state])

print("Starting LunarLander Monte Carlo training...")
print(f"State space size: {TOTAL_STATES:,}")
print(f"Total episodes to train: {NUM_EPISODES:,}")
print("=" * 50)

Starting LunarLander Monte Carlo training...
State space size: 1,048,576
Total episodes to train: 30,000


In [25]:
# --------------------------------
# Monte Carlo Training
# --------------------------------

for episode in range(NUM_EPISODES):
  #Generate episode
  episode_states = []
  episode_actions = []
  episode_rewards = []

  state,_ = env.reset()
  state = discretize(state)
  done = False
  steps = 0

  #Collect episode data
  while not done and steps < 1000:
    action = choose_action(state,EPSILON)
    next_state, reward, done, truncated, _ = env.step(action)

    episode_states.append(state)
    episode_actions.append(action)
    episode_rewards.append(reward)

    state = discretize(next_state)
    steps += 1

  #Calculate returns for this episode

  episode_length = len(episode_rewards)
  total_reward = sum(episode_rewards)
  all_rewards.append(total_rewards)

  #Calculate discounted returns (working backwards)
  G = 0
  for t in range(episode_length - 1, -1, -1):
    G = GAMMA * G + episode_rewards[t]

    state_t = episode_states[t]
    action_t = episode_actions[t]

    returns_sum[state_t, action_t] += G
    returns_count[state_t, action_t] +=1

    #Update Q-value with average return
    Q[state_t, action_t] = returns_sum[state_t, action_t] / returns_count[state_t, action_t]

  #Epsilon decay
  EPSILON = max(EPSILON * EPSILON_DECAY, EPSILON_MIN)

  #Logging
  if episode % 50 == 0:
    writer.add_scalar("Train/EpisodeReward", total_reward, episode)
    if episode >= 100:
      avg_reward = np.mean(all_rewards[-100:])
      writer.add_scalar("Train/MovingAverageReward", avg_reward, episode)
      writer.add_scalar("Train/Epsilon", EPSILON, episode)

  #Simple progress tracking
  if (episode + 1) % 1000 == 0:
    recent_avg = np.mean(all_rewards[-100:]) if len(all_rewards) >= 100 else np.mean(all_rewards)
    states_visited = np.sum(returns_count > 0)
    print(f"Episode {episode + 1:5d} / {NUM_EPISODES} - Avg Reward: {recent_avg:.1f} - States Visited: {states_visited}")

env.close()

print("Training completed! Starting evaluation...")

Episode  1000 / 30000 - Avg Reward: -322.0 - States Visited: 3823
Episode  2000 / 30000 - Avg Reward: -322.0 - States Visited: 4908
Episode  3000 / 30000 - Avg Reward: -322.0 - States Visited: 5783
Episode  4000 / 30000 - Avg Reward: -322.0 - States Visited: 6980
Episode  5000 / 30000 - Avg Reward: -322.0 - States Visited: 8029
Episode  6000 / 30000 - Avg Reward: -322.0 - States Visited: 8582
Episode  7000 / 30000 - Avg Reward: -322.0 - States Visited: 9037
Episode  8000 / 30000 - Avg Reward: -322.0 - States Visited: 9380
Episode  9000 / 30000 - Avg Reward: -322.0 - States Visited: 9747
Episode 10000 / 30000 - Avg Reward: -322.0 - States Visited: 10012
Episode 11000 / 30000 - Avg Reward: -322.0 - States Visited: 10377
Episode 12000 / 30000 - Avg Reward: -322.0 - States Visited: 10645
Episode 13000 / 30000 - Avg Reward: -322.0 - States Visited: 10886
Episode 14000 / 30000 - Avg Reward: -322.0 - States Visited: 11042
Episode 15000 / 30000 - Avg Reward: -322.0 - States Visited: 11189
Epis

In [26]:
# --------------------------------------
# Testing Phase
# --------------------------------------

test_env = gym.make("LunarLander-v3", render_mode = "rgb_array")
test_env = RecordVideo(
    test_env,
    video_folder = "lunarlander_videos_mc",
    episode_trigger = lambda ep_id: ep_id < 5,
    name_prefix = "mc_lander"
)

total_test_rewards = []
successful_landings = 0

for test_episode in range(50):
  state,_ = test_env.reset()
  state = discretize(state)
  done = False
  test_reward = 0
  steps = 0

  while not done and steps < 1000:
    action = np.argmax(Q[state])
    next_state, reward, done, truncated, _ = test_env.step(action)
    test_reward += reward
    state = discretize(next_state)
    steps += 1

  total_test_rewards.append(test_reward)
  if test_reward >= 200:
    successful_landings += 1

  if test_episode % 10 == 0:
    writer.add_scalar("Test/EpisodeReward", test_reward, test_episode)


#Calculate final statistics
avg_test_reward = np.mean(total_test_rewards)
success_rate = successful_landings / len(total_test_rewards) * 100
final_training_avg = np.mean(all_rewards[-100:]) if len(all_rewards) >= 100 else np.mean(all_rewards)
states_visited = np.sum(returns_count > 0)

writer.add_scalar("Test/AverageReward", avg_test_reward, 0)
writer.add_scalar("Test/SuccessRate", success_rate, 0)
writer.add_scalar("Final/TrainingAverage", final_training_avg, 0)
writer.add_scalar("Final/StatesVisited", states_visited, 0)

print(f"\n=== MONTE CARLO RESULTS ===")
print(f"Training Episodes: {NUM_EPISODES:,}")
print(f"Final Training Average: {final_training_avg: .2f}")
print(f"States Visited: {states_visited:,} / {TOTAL_STATES:,}")
print(f"Average Test Reward: {avg_test_reward:.2f}")
print(f"Successful Landings: {successful_landings}/{len(total_test_rewards)} ({success_rate:.1f}%)")
print(f"Best Test Episode: {max(total_test_rewards):.2f}")

test_env.close()
writer.close()

print(f"\nTensorBoard logs: runs1/lunarlander_mc")
print(f"Videos saved to: lunarlander_videos_mc/")

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"



=== MONTE CARLO RESULTS ===
Training Episodes: 30,000
Final Training Average: -322.03
States Visited: 13,245 / 1,048,576
Average Test Reward: -28.55
Successful Landings: 0/50 (0.0%)
Best Test Episode: 133.08

TensorBoard logs: runs1/lunarlander_mc
Videos saved to: lunarlander_videos_mc/
